# 03 — Mapping Layer (Logical Targets → CSS Selectors)

This notebook loads the IR produced by Notebook 01 and maps logical element
names (e.g., `new_todo_input`, `todo_item`) to real CSS selectors for the
TodoMVC application.

Pipeline:
NL → IR → Mapped IR → Code → Execution → Analysis


In [7]:
from pathlib import Path
import json


## 1. Load IR from Disk


In [8]:
ir_path = Path("../sample-data/ir_examples/todomvc_ir.json")

with open(ir_path, "r") as f:
    ir = json.load(f)

ir


{'test_name': 'todomvc_add_and_complete_items',
 'description': 'Add “buy milk”, add “get gas in Susie’s car”, then complete “buy milk”, and complete “get gas in Susie’s car”.\n',
 'steps': [{'action': 'input',
   'target': 'new_todo_input',
   'value': 'buy milk'},
  {'action': 'press', 'target': 'new_todo_input', 'key': 'Enter'},
  {'action': 'input',
   'target': 'new_todo_input',
   'value': 'get gas in Susie’s car'},
  {'action': 'press', 'target': 'new_todo_input', 'key': 'Enter'},
  {'action': 'click', 'target': 'todo_item_1_checkbox'},
  {'action': 'click', 'target': 'todo_item_2_checkbox'}],
 'metadata': {'priority': 'high'}}

## 2. Define the TodoMVC Mapping Model

This maps logical names → real CSS selectors.


In [9]:
APP_MODEL = {
    "new_todo_input": "input.new-todo",
    "todo_item": ".todo-list li",
    "toggle_checkbox": ".toggle",
    "delete_button": ".destroy",
    "todo_count": ".todo-count",
    "filter_all": "a[href='#/']",
    "filter_active": "a[href='#/active']",
    "filter_completed": "a[href='#/completed']"
}


## 3. Apply Mapping to IR Steps


In [10]:
def map_ir_targets(ir, model):
    mapped_steps = []

    for step in ir["steps"]:
        mapped_step = step.copy()

        target = step.get("target")
        if target in model:
            mapped_step["selector"] = model[target]
        else:
            mapped_step["selector"] = None  # unmapped target

        mapped_steps.append(mapped_step)

    mapped_ir = ir.copy()
    mapped_ir["steps"] = mapped_steps
    return mapped_ir


## 4. Generate Mapped IR


In [11]:
mapped_ir = map_ir_targets(ir, APP_MODEL)
mapped_ir


{'test_name': 'todomvc_add_and_complete_items',
 'description': 'Add “buy milk”, add “get gas in Susie’s car”, then complete “buy milk”, and complete “get gas in Susie’s car”.\n',
 'steps': [{'action': 'input',
   'target': 'new_todo_input',
   'value': 'buy milk',
   'selector': 'input.new-todo'},
  {'action': 'press',
   'target': 'new_todo_input',
   'key': 'Enter',
   'selector': 'input.new-todo'},
  {'action': 'input',
   'target': 'new_todo_input',
   'value': 'get gas in Susie’s car',
   'selector': 'input.new-todo'},
  {'action': 'press',
   'target': 'new_todo_input',
   'key': 'Enter',
   'selector': 'input.new-todo'},
  {'action': 'click', 'target': 'todo_item_1_checkbox', 'selector': None},
  {'action': 'click', 'target': 'todo_item_2_checkbox', 'selector': None}],
 'metadata': {'priority': 'high'}}

## 5. Save Mapped IR for Downstream Notebooks


In [12]:
output_path = Path("../sample-data/ir_examples/todomvc_mapped_ir.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(mapped_ir, f, indent=4)

output_path


PosixPath('../sample-data/ir_examples/todomvc_mapped_ir.json')